In [1]:
import sys
from pathlib import Path
# notebooks/ is one level below the project root; src/ is at the same level
ROOT = Path.cwd().parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))


# Task 1: Data Handling & Memory Management

This notebook demonstrates efficient loading and processing of the Telecom Italia Milan dataset (~5 GB across 62 tab-separated daily files). All configurable parameters live in `config.py`; all parsing logic lives in `preprocessing.py`.

## Column Schema (corrected field order per [1])

| Index | Field | Type | Description |
|-------|-------|------|-------------|
| 0 | `square_id` | int | Grid cell identifier (1–10,000) |
| 1 | `time_interval` | int | Start of 10-min slot (Unix milliseconds) |
| 2 | `country_code` | int | Origin country phone code |
| 3 | `sms_in` | float | Received SMS activity (nullable) |
| 4 | `sms_out` | float | Sent SMS activity (nullable) |
| 5 | `call_in` | float | Received call activity (nullable) |
| 6 | `call_out` | float | Issued call activity (nullable) |
| 7 | `internet_traffic` | float | Internet CDR count — **our target** (nullable) |

Only columns 0, 1, and 7 are relevant to this assignment [1]. All others are discarded at parse time.

> **Note**: Multiple rows share the same `(square_id, time_interval)` — one per originating country code. We **sum** internet activity across country codes to get the area-level total.

**References**

[1] G. Barlacchi et al., \"A multi-source dataset of urban life in the city of Milan and the Province of Trentino,\" *Sci. Data*, vol. 2, p. 150055, 2015. doi:10.1038/sdata.2015.55

## 1.1 Hardware & Software Setup

In [2]:
import platform, psutil, sys
import pandas as pd
import numpy as np
import time
from pathlib import Path

import config
import preprocessing as pp

print("=== Hardware & Software Environment ===")
print(f"OS             : {platform.system()} {platform.release()}")
print(f"Processor      : {platform.processor()}")
mem = psutil.virtual_memory()
print(f"RAM (total)    : {mem.total / 1e9:.1f} GB")
print(f"RAM (available): {mem.available / 1e9:.1f} GB")
print(f"Python         : {sys.version.split()[0]}")
print(f"pandas         : {pd.__version__}")
print(f"numpy          : {np.__version__}")
print()
print(f"Train window   : {config.TRAIN_START.date()} → {config.TRAIN_END.date()}")
print(f"Test window    : {config.TEST_START.date()} → {config.TEST_END.date()}")

=== Hardware & Software Environment ===
OS             : Windows 11
Processor      : Intel64 Family 6 Model 170 Stepping 4, GenuineIntel
RAM (total)    : 16.8 GB
RAM (available): 1.1 GB
Python         : 3.12.6
pandas         : 3.0.3
numpy          : 2.4.5

Train window   : 2013-11-01 → 2013-12-15
Test window    : 2013-12-16 → 2013-12-22


## 1.2 Locate Data Files

In [3]:
# Files may be in data/ subdirectory or the project root
files = pp.find_raw_files(config.DATA_DIR)
if not files:
    files = pp.find_raw_files(config.PROJECT_DIR)

print(f"Found {len(files)} / 62 data files")
for f in files[:5]:
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.0f} MB)")
if len(files) > 5:
    print(f"  ... and {len(files)-5} more")

if not files:
    raise FileNotFoundError(
        "No data files found. Download from Harvard Dataverse "
        "(doi:10.7910/DVN/EGZHFV) and place in the data/ directory."
    )
if len(files) < 52:
    print(f"\nWARNING: {len(files)} files present. "
          "Task 3 requires files covering Nov 1 through Dec 22 (~52 files minimum).")

Found 62 / 62 data files
  sms-call-internet-mi-2013-11-01.txt  (323 MB)
  sms-call-internet-mi-2013-11-02.txt  (315 MB)
  sms-call-internet-mi-2013-11-03.txt  (313 MB)
  sms-call-internet-mi-2013-11-04.txt  (361 MB)
  sms-call-internet-mi-2013-11-05.txt  (373 MB)
  ... and 47 more


## 1.3 Baseline: Naive Load (No Optimisation)

Load a single file with default pandas settings — all 8 columns, default `float64`/`int64` dtypes, no aggregation — to establish a memory baseline.

In [4]:
sample_file = files[0]

t0 = time.time()
df_naive = pd.read_csv(
    sample_file,
    sep="\t",
    header=None,
    names=pp._RAW_COLUMN_NAMES,
)
t_naive = time.time() - t0

mem_naive = pp.memory_profile(df_naive, "Naive (all columns, default dtypes)")
print(f"File          : {sample_file.name}")
print(f"Rows          : {mem_naive['rows']:,}")
print(f"Columns       : {mem_naive['columns']}")
print(f"Memory        : {mem_naive['memory_mb']:.1f} MB")
print(f"Load time     : {t_naive:.2f}s")
print()
print("Dtypes (naive):")
print(df_naive.dtypes)
print()
print(df_naive.head(3))

File          : sms-call-internet-mi-2013-11-01.txt
Rows          : 4,842,625
Columns       : 8
Memory        : 295.6 MB
Load time     : 2.54s

Dtypes (naive):
square_id             int64
time_interval         int64
country_code          int64
sms_in              float64
sms_out             float64
call_in             float64
call_out            float64
internet_traffic    float64
dtype: object

   square_id  time_interval  country_code    sms_in   sms_out   call_in  \
0          1  1383260400000             0  0.081363       NaN       NaN   
1          1  1383260400000            39  0.141864  0.156787  0.160938   
2          1  1383261000000             0  0.136588       NaN       NaN   

   call_out  internet_traffic  
0       NaN               NaN  
1  0.052275         11.028366  
2  0.027300               NaN  


## 1.4 Optimisation Strategy

Four complementary techniques reduce memory consumption. All are implemented in `preprocessing.py`.

| Technique | Mechanism | Saving |
|-----------|-----------|--------|
| **Column selection** (`usecols`) | Read only `square_id`, `time_interval`, `internet_traffic` — 3 of 8 columns | ~63% fewer columns |
| **Robust parsing then downcast** | Read as `string`, validate with `pd.to_numeric(errors='coerce')`, then cast to `uint16` / `float32` | ~50–75% per column vs naive; skips bad rows cleanly |
| **Aggregation at load time** | Sum `internet_traffic` per `(square_id, time_interval)` — collapses country-code rows | ~5–10× row reduction |
| **Chunked reading** | `chunksize=200_000` keeps peak memory bounded to one chunk at a time | Enables processing files larger than available RAM |

### Why `uint16` instead of `int32` for `square_id`
Square IDs range from 1–10,000. `uint16` holds up to 65,535, using 2 bytes per value versus 4 bytes for `int32` — a further 50% saving on that column.

### Why read as `string` first?
The raw files are sparse: rows have between 4 and 8 tab-separated fields. Reading as `string` then converting with `errors='coerce'` allows invalid or missing values to be identified and counted explicitly (`MissingDataStats`) rather than silently corrupting the numeric output.

In [5]:
# Collect all chunks from one file and merge, to compare against naive
t0 = time.time()
chunks_opt = []
for chunk, stats in pp.iter_file_chunks(sample_file, config.CHUNKSIZE):
    if not chunk.empty:
        chunks_opt.append(chunk)

df_opt = pd.concat(chunks_opt, ignore_index=True)
# Final aggregation across chunks (country codes split across chunk boundaries)
df_opt = (
    df_opt.groupby(["square_id", "time_interval"], as_index=False, observed=True)
    ["internet_traffic"].sum()
)
t_opt = time.time() - t0

mem_opt = pp.memory_profile(df_opt, "Optimised (3 cols, uint16/float32, aggregated)")
print(f"Rows          : {mem_opt['rows']:,}")
print(f"Columns       : {mem_opt['columns']}")
print(f"Memory        : {mem_opt['memory_mb']:.1f} MB")
print(f"Load time     : {t_opt:.2f}s")
print()
print("Dtypes (optimised):")
print(df_opt.dtypes)
print()
print(df_opt.head(3))

Rows          : 1,439,982
Columns       : 3
Memory        : 19.2 MB
Load time     : 9.22s

Dtypes (optimised):
square_id            uint16
time_interval         int64
internet_traffic    float32
dtype: object

   square_id  time_interval  internet_traffic
0          1  1383260400000         11.028366
1          1  1383261000000         11.127101
2          1  1383261600000         10.892771


In [6]:
# --- Memory comparison report ---
mem_red  = (1 - mem_opt['memory_mb'] / mem_naive['memory_mb']) * 100
row_red  = (1 - mem_opt['rows'] / mem_naive['rows']) * 100

comparison = pd.DataFrame([
    mem_naive,
    mem_opt,
]).set_index("label")

print("=" * 65)
print("Memory Optimisation Report — single file:", sample_file.name)
print("=" * 65)
print(comparison.to_string())
print("-" * 65)
print(f"Memory reduction : {mem_red:.1f}%")
print(f"Row reduction    : {row_red:.1f}%  (country-code aggregation)")
print("=" * 65)

Memory Optimisation Report — single file: sms-call-internet-mi-2013-11-01.txt
                                                   rows  columns  memory_mb
label                                                                      
Naive (all columns, default dtypes)             4842625        8    295.570
Optimised (3 cols, uint16/float32, aggregated)  1439982        3     19.226
-----------------------------------------------------------------
Memory reduction : 93.5%
Row reduction    : 70.3%  (country-code aggregation)


## 1.5 Data Quality Report

`preprocessing.py` tracks data quality issues while reading via `MissingDataStats`.

In [7]:
print(f"Data quality stats for {sample_file.name}:")
print(f"  Rows read                       : {stats.rows_parsed:,}")
print(f"  Missing internet_traffic values : {stats.blank_traffic_values:,}  "
      f"(treated as 0.0 — no CDR = no activity)")
print(f"  Invalid square_ids              : {stats.bad_square_ids:,}")
print(f"  Invalid timestamps              : {stats.bad_timestamps:,}")
print(f"  Invalid activity values         : {stats.bad_traffic_values:,}")
print(f"  Duplicate (square, time) rows   : {stats.country_code_rows_collapsed:,}  "
      f"(collapsed by summing across country codes)")

Data quality stats for sms-call-internet-mi-2013-11-01.txt:
  Rows read                       : 4,842,625
  Missing internet_traffic values : 2,488,626  (treated as 0.0 — no CDR = no activity)
  Invalid square_ids              : 0
  Invalid timestamps              : 0
  Invalid activity values         : 0
  Duplicate (square, time) rows   : 3,402,629  (collapsed by summing across country codes)


## 1.6 Process All Files → Parquet

Two outputs are produced:
- **`cell_totals.csv`** — total 2-month internet traffic per grid square (all 10,000); used for the Task 2 PDF and heatmap
- **`area_timeseries.parquet`** — full 10-minute time series for the three target areas only; used for Tasks 2 and 3

In [8]:
CELL_TOTALS_PATH   = config.PROCESSED_DIR / "cell_totals.csv"
TIMESERIES_PATH    = config.PROCESSED_DIR / "area_timeseries.parquet"
STATS_PATH         = config.TABLES_DIR / "parse_quality_log.csv"

if CELL_TOTALS_PATH.exists() and TIMESERIES_PATH.exists():
    print("Processed files already exist. Loading...")
    cell_totals    = pd.read_csv(CELL_TOTALS_PATH)
    area_timeseries = pd.read_parquet(TIMESERIES_PATH)
    target_areas   = pd.read_csv(config.PROCESSED_DIR / "target_areas.csv")
else:
    print("Building cell totals across all files...")
    t0 = time.time()
    cell_totals, stats_df = pp.compute_cell_totals(files, config.CHUNKSIZE)
    print(f"  Done in {time.time()-t0:.1f}s — {len(cell_totals):,} grid squares")

    target_areas = pp.identify_target_areas(cell_totals, config.FIXED_SQUARE_IDS)
    print()
    print("Target areas:")
    print(target_areas.to_string(index=False))

    target_ids = target_areas["square_id"].tolist()

    print()
    print("Building target area time series...")
    t0 = time.time()
    area_timeseries, gap_df = pp.build_area_timeseries(
        files, target_ids, config.CHUNKSIZE, config.DATA_FREQUENCY
    )
    print(f"  Done in {time.time()-t0:.1f}s — {len(area_timeseries):,} rows")
    print()
    print("Missing interval fill counts:")
    print(gap_df.to_string(index=False))

    city_heatmap = pp.build_heatmap_grid(cell_totals, config.GRID_SIZE)

    pp.save_outputs(
        cell_totals, target_areas, area_timeseries,
        city_heatmap, stats_df,
        config.PROCESSED_DIR, config.TABLES_DIR,
    )
    print(f"\nSaved outputs to {config.PROCESSED_DIR}/")

Building cell totals across all files...
  Done in 510.7s — 10,000 grid squares

Target areas:
          label  square_id
highest_traffic       5161
     fixed_4159       4159
     fixed_4556       4556

Building target area time series...
  Done in 605.0s — 26,784 rows

Missing interval fill counts:
 square_id  n_filled_gaps
      5161           1440
      4159           1440
      4556           1440

Saved outputs to C:\Users\icyez\Downloads\Telecom\processed/


In [9]:
print("=== Processed Data Summary ===")
print()
print("cell_totals:")
print(pp.memory_profile(cell_totals, "cell_totals"))
print(cell_totals.describe().round(2))
print()
print("area_timeseries:")
print(pp.memory_profile(area_timeseries, "area_timeseries"))
print(area_timeseries.head(3))

=== Processed Data Summary ===

cell_totals:
{'label': 'cell_totals', 'rows': 10000, 'columns': 2, 'memory_mb': 0.095}
       square_id  total_traffic
count   10000.00       10000.00
mean     5000.50      490049.48
std      2886.90      805247.19
min         1.00         213.63
25%      2500.75      102422.97
50%      5000.50      243144.64
75%      7500.25      500834.04
max     10000.00    11114599.13

area_timeseries:
{'label': 'area_timeseries', 'rows': 26784, 'columns': 3, 'memory_mb': 0.358}
   square_id            datetime  internet_traffic
0       5161 2013-10-31 23:00:00        379.972931
1       5161 2013-10-31 23:10:00        371.122711
2       5161 2013-10-31 23:20:00        429.297943


## 1.7 Summary of Optimisation Impact

| Technique | Memory impact | Correctness impact |
|-----------|--------------|--------------------|
| `usecols` (3 of 8) | ~63% fewer bytes per row | None — SMS/call columns not needed |
| `string` → `uint16`/`float32` | ~50–75% per column vs default int64/float64 | Safer — invalid rows counted, not silently corrupted |
| Country-code aggregation | ~5–10× row reduction | Required — multiple rows per (area, time) are physically one observation |
| Chunked reading | Peak RAM ≈ 1 chunk, not 1 file | Enables files larger than available RAM |
| Parquet output | ~10× smaller on disk than raw TSV | Fast columnar re-reads in notebooks 2 and 3 |

## 1.8 Challenges

1. **Variable-width rows**: Rows have 4–8 fields depending on which CDR types were present. Reading as `string` with `on_bad_lines='skip'` handles this without assumptions about column count.
2. **Country-code multiplicity**: Each `(square_id, time_interval)` pair appears once per originating country — summing across these is semantically correct, not an approximation.
3. **Field ordering error in [1]**: The paper states `country_code` as the second field; it is actually the third (index 2), shifting all subsequent fields.
4. **Scale**: ~300 MB × 62 files ≈ 18 GB if loaded naively. The chunked per-file approach keeps peak RAM to ~one file at a time.